# 3 · Moving house: editing relations

**The problem.** Alice moves. Her old home stays in the book, because her lodger still lives there, but her link to
it must go and a new one must appear. Her work address changes label. A new colleague joins who is "just like
Alice" apart from the name. Each of these is an ordinary edit, and each shows something about how the framework
treats relations as shared facts rather than private arrays.

This case study reuses case study 2's address book. The schemas are repeated in one cell so the notebook stands on
its own.

In [1]:
import { Proxies, Schemas } from "@mbse/schemas/Framework";
import type { OfAdjacency, OfEntry } from "@mbse/schemas/Framework/Visitors";
import { entries } from "./toolkit.js";

const store = new Proxies.OfStore();

let Contact = new Schemas.OfObject.Builder().ref()
  .properties(
    (p) => p.name("given_name").of((t) => t.as_native(String)),
    (p) => p.name("family_name").of((t) => t.as_native(String)),
  )
  .create();
let Address = new Schemas.OfObject.Builder().ref()
  .properties(
    (p) => p.name("street").of((t) => t.as_native(String)),
    (p) => p.name("locality").of((t) => t.as_native(String)),
  )
  .create();
const ContactAddresses = new Schemas.OfRelation.Builder()
  .links("contact", "address")
  .properties((p) => p.name("label").of((t) => t.as_native(String)))
  .create();
Contact = new Schemas.OfObject.Builder(Contact).relations((adj) => adj.name("addresses").of(ContactAddresses).me("contact")).update();
Address = new Schemas.OfObject.Builder(Address).relations((adj) => adj.name("residents").of(ContactAddresses).me("address")).update();
store.register("Contact", Contact);
store.register("Address", Address);
store.register("ContactAddresses", ContactAddresses);

function show(obj: Proxies.Instance, adjacency: string, link: string, field: string): string[][] {
  return entries(obj, adjacency).map((e) => [e[link][field], e.label ?? "-"]).sort();
}

In [2]:
const oldHome = store.Address().street("7 Old Road").locality("Oxford").create();
const office = store.Address().street("1 Market Square").locality("Oxford").create();

const alice = store.Contact()
  .given_name("Alice")
  .addresses((x: any) => x.address(oldHome).label("home"))
  .addresses((x: any) => x.address(office).label("work"))
  .create();
const lodger = store.Contact().given_name("Lewis").addresses((x: any) => x.address(oldHome).label("home")).create();

console.log("alice:   ", show(alice, "addresses", "address", "street"));
console.log("old home:", show(oldHome, "residents", "contact", "given_name"));

alice:    [ [ "1 Market Square", "work" ], [ "7 Old Road", "home" ] ]


old home: [ [ "Alice", "home" ], [ "Lewis", "home" ] ]


## Removing an entry

There's no `alice.addresses.remove(...)`, for the same reason as in case study 2: an entry isn't owned by Alice. You
remove it through a builder, by visiting the adjacency and handing back the entry to drop. The builder's
`adjacency(name, callback)` gives you the adjacency handle `a`. `a.entries(...)` visits each entry, and
`a.remove(entry)` drops one. Nothing takes effect until `update()`:

In [3]:
function isLinkTo(entry: OfEntry, link: string, target: Proxies.Instance): boolean {
  let found = false;
  entry.link(link, (k) => k.target((t) => (found = t === target)));
  return found;
}

store.Contact(alice)
  .adjacency("addresses", (a: OfAdjacency) => a.entries((e) => (isLinkTo(e, "address", oldHome) ? a.remove(e) : undefined)))
  .addresses((x: any) => x.address((n: any) => n.street("2 New Lane").locality("Bath")).label("home"))
  .update();

console.log("alice:   ", show(alice, "addresses", "address", "street"));
console.log("old home:", show(oldHome, "residents", "contact", "given_name"));

alice:    [ [ "1 Market Square", "work" ], [ "2 New Lane", "home" ] ]


old home: [ [ "Lewis", "home" ] ]


One `update()` removed the old fact and added the new one. Because an entry is one fact, it also left the old
address's `residents`. Lewis is still there, untouched, and the old address object still exists. Only the link
went.

### Why the callback style for removal?

`a.entries(...)` hands you the entry *handles*, and `a.remove(...)` takes one back. You never get an array you could
keep, mutate and forget to write back. The builder decides what "remove" means, here dropping it from its working
copy until `update()`. A serializer or validator implementing the same protocol would do something else with the
same calls.

Filtering by what an entry *contains* is common enough that `toolkit.remove_entries` wraps it, using the same rows
as `entries`:

In [4]:
import { remove_entries } from "./toolkit.js";

remove_entries(lodger, "addresses", (row) => row.address === oldHome);
console.log("old home:", show(oldHome, "residents", "contact", "given_name"));

old home: []


## Changing an entry's property

An entry's properties are part of its identity: `(alice, office, "work")` and `(alice, office, "office")` are two
different facts, as you saw with duplicate elision. So "relabeling" is removing one fact and adding another, in one
builder:

In [5]:
store.Contact(alice)
  .adjacency("addresses", (a: OfAdjacency) => a.entries((e) => (isLinkTo(e, "address", office) ? a.remove(e) : undefined)))
  .addresses((x: any) => x.address(office).label("office"))
  .update();
console.log(show(alice, "addresses", "address", "street"));

[ [ "1 Market Square", "office" ], [ "2 New Lane", "home" ] ]


## Cloning copies the relations too

A new colleague, Bob, works at the same office and lives at the same place as Alice. `clone()` copies properties
*and* entries, with the clone in the place of the original:

In [6]:
const bob = store.Contact(alice).given_name("Bob").clone();

console.log("bob:   ", show(bob, "addresses", "address", "street"));
console.log("office:", show(office, "residents", "contact", "given_name"));

bob:    [ [ "1 Market Square", "office" ], [ "2 New Lane", "home" ] ]


office: [ [ "Alice", "office" ], [ "Bob", "office" ] ]


Notice what cloning means for the *other* end: the office now has two residents. That's the only consistent reading
of "a copy of Alice's facts with Bob in her place". Cloning a contact with no relations is just a copy of its
properties.

## What `update()` replaces

A builder started from an instance works on a **snapshot** of that instance, its properties and all of its entries.
`update()` writes the whole snapshot back: it drops every entry the object had and adds the builder's. That makes an
update all-or-nothing and easy to reason about. It also means two builders open on the same object at once don't
merge. The last `update()` wins:

In [7]:
const first = store.Contact(bob).addresses((x: any) => x.address(oldHome).label("weekend"));
const second = store.Contact(bob).given_name("Robert");

first.update();
console.log("after first: ", show(bob, "addresses", "address", "street"));
second.update(); // started before `first` was written, so it doesn't contain the weekend entry
console.log("after second:", show(bob, "addresses", "address", "street"), bob.given_name);

after first:  [
  [ "1 Market Square", "office" ],
  [ "2 New Lane", "home" ],
  [ "7 Old Road", "weekend" ]
]


after second: [ [ "1 Market Square", "office" ], [ "2 New Lane", "home" ] ] Robert


**The rule:** open a builder, make the change, finish it. Don't keep builders around across other edits. (Whether
`update()` should instead merge is an open design question, recorded as finding F4 in the test plan. The behavior
above is what exists today.)

## Leaving an adjacency alone

The flip side: a builder that doesn't touch an adjacency writes back exactly the entries it read, so a
properties-only edit never loses relations:

In [8]:
store.Contact(alice).family_name("Liddell").update();
console.log(show(alice, "addresses", "address", "street"));

[ [ "1 Market Square", "office" ], [ "2 New Lane", "home" ] ]


## What you learned

- Entries are removed through a builder: `.adjacency(name, (a: OfAdjacency) => a.entries((e) => a.remove(e) ...))`,
  then `update()`.
- An entry's properties are part of the fact, so changing one means remove and add.
- Removing an entry affects both ends, and never deletes the linked objects.
- `clone()` copies entries with the clone in place of the original, so other ends see the clone too.
- `update()` writes back the builder's whole snapshot. Keep builders short-lived.

**Next:** so far every relation linked two *different* kinds of object. Case study 4 models a family tree, where
people link to people, cycles appear, and "everything connected to X" becomes a question worth asking.